### Reconstruction and Coregistration

####  *AUTHOR:* Ehsan Farahbakhsh
####  *CONTACT:* e.farahbakhsh@sydney.edu.au
####  *DATE last modified:* 04/05/2026

This notebook generates one-sided buffer zones around trench lines within subduction zones at each time step and creates both random (unlabelled) points and target points within these buffers. It also reconstructs the locations of known mineral occurrences (porphyry deposits and prospects). All three point sets — reconstructed mineral occurrences, random points, and target points — are then coregistered with the nearest point on the trench lines, and the feature values calculated for that trench point are assigned to the corresponding points in each set.

We begin by importing the required libraries:

In [ ]:
# Import libraries
from ipywidgets import interact
import os

import cartopy.crs as ccrs
import cmcrameri.cm as ccm
import gplately
from gplately import PlateModelManager, PlateReconstruction, PlotTopologies
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import pandas as pd

# Note: Ensure the 'lib' folder is located in the same folder as this notebook
from lib.reconstruction_coregistration import *
from lib.feature_extraction import *
from lib.plot import *

# Load configuration parameters (e.g., paths, model names)
from parameters import parameters

### Setup

As defined in `parameters.py`, the cell below configures the analysis parameters and specifies the paths to input/output files and directories. You can also specify the number of cores to use by setting an appropriate value for the `n_jobs` variable at the end of the cell.

**Note:** You can modify the analysis settings directly in `parameters.py`, located in the same folder as this notebook. The file is structured as a dictionary; look for keys such as `timespan` and `grid_resolution` to adjust their values as needed.

In [ ]:
# Set up temporal analysis parameters
plate_model = parameters["plate_model"]
anchor_plate_id = parameters["anchor_plate_id"]

temporal_resolution = parameters["temporal_resolution"]
time_min = parameters["timespan"]["min"] # Youngest time to analyse
time_max = parameters["timespan"]["max"] # Oldest time to analyse
# Create an array of time steps for analysis (e.g., 0, 1, 2, ... Ma)
time_steps = range(time_min, time_max + temporal_resolution, temporal_resolution)

buffer_distance = parameters["buffer_distance"]
num_random = parameters["num_random"] # Number of random points to be generated at each time step
grid_resolution = parameters["grid_resolution"]
columns_to_drop_deposit = parameters["columns_to_drop_deposit"]
columns_to_drop_unlabelled = parameters["columns_to_drop_unlabelled"]
columns_to_drop_backarc = parameters["columns_to_drop_backarc"]

# Directory paths for inputs and outputs
inputs_dir = parameters["inputs_dir"]
outputs_dir = parameters["outputs_dir"]
buffer_zones_dir = parameters["buffer_zones_dir"]
continents_recon_dir = parameters["continents_recon_dir"]

# Data filenames
subduction_data_filename = parameters["subduction_data_filename"]
deposit_coords_filename = parameters["deposit_coords_filename"]
deposit_coords_recon_filename = parameters["deposit_coords_recon_filename"]
deposit_coords_recon_all_filename = parameters["deposit_coords_recon_all_filename"]
deposit_coords_recon_all_filtered_filename = parameters["deposit_coords_recon_all_filtered_filename"]
continents_recon_filename = parameters["continents_recon_filename"]
unlabelled_coords_filename = parameters["unlabelled_coords_filename"]
backarc_coords_filename = parameters["backarc_coords_filename"]
deposit_data_filename = parameters["deposit_data_filename"]
unlabelled_data_filename = parameters["unlabelled_data_filename"]
backarc_data_filename = parameters["backarc_data_filename"]

buffer_zones_dir = os.path.join(outputs_dir, buffer_zones_dir)
continents_recon_dir = os.path.join(outputs_dir, continents_recon_dir)

# Construct full file paths
subduction_data_filename = os.path.join(outputs_dir, subduction_data_filename)
deposit_coords_filename = os.path.join(inputs_dir, deposit_coords_filename)
deposit_coords_recon_filename = os.path.join(outputs_dir, deposit_coords_recon_filename)
deposit_coords_recon_all_filename = os.path.join(outputs_dir, deposit_coords_recon_all_filename)
deposit_coords_recon_all_filtered_filename = os.path.join(outputs_dir, deposit_coords_recon_all_filtered_filename)
continents_recon_filename = os.path.join(plate_model, "ContinentalPolygons", continents_recon_filename)
unlabelled_coords_filename = os.path.join(outputs_dir, unlabelled_coords_filename)
backarc_coords_filename = os.path.join(outputs_dir, backarc_coords_filename)
deposit_data_filename = os.path.join(outputs_dir, deposit_data_filename)
unlabelled_data_filename = os.path.join(outputs_dir, unlabelled_data_filename)
backarc_data_filename = os.path.join(outputs_dir, backarc_data_filename)

# Paths to different types of grids
agegrid_dir = os.path.join(inputs_dir, "SeafloorAge")
crusthick_dir = os.path.join(inputs_dir, "CrustalThickness")

# Number of cores to be used for running this notebook
n_jobs = 20

The cell below loads the necessary files to create the `PlateReconstruction` and `PlateTopologies` objects, which will later be used for reconstruction and generating visualisations. Moreover, the feature values calculated using the `01_feature_extraction` notebook will also be loaded in this cell.

In [ ]:
# Plate motion model
pmm = PlateModelManager()
pm = pmm.get_model(plate_model)

rotation_model = pm.get_rotation_model()
topology_features = pm.get_topologies()

static_polygons = pm.get_static_polygons()
coastlines = pm.get_coastlines()
continents = pm.get_continental_polygons()
COBs = pm.get_COBs()

plate_reconstruction = PlateReconstruction(
    rotation_model=rotation_model,
    topology_features=topology_features,
    static_polygons=static_polygons,
    anchor_plate_id=anchor_plate_id,
)

gplot = PlotTopologies(
    plate_reconstruction=plate_reconstruction,
    coastlines=coastlines,
    continents=continents,
    COBs=COBs,
    anchor_plate_id=anchor_plate_id,
)

subduction_data = pd.read_csv(subduction_data_filename) # Load feature values
deposit_coords = pd.read_csv(deposit_coords_filename) # Load deposit coordinates and weights

# Define map projection (Mollweide provides a good global view)
projection = ccrs.Mollweide(central_longitude=30)

### Buffer Zones

This cell creates one-sided buffer zones around trench lines within subduction zones at each time step, covering arc–backarc environments. The buffer width can be customised using the `buffer_distance` argument in the function (default: 6°). You can also adjust this value in the `parameters.py` file. In the following cells, random (unlabeled) points and target points are generated within these buffer zones.

In [ ]:
# Create buffer zones if the relevant directory does not exist
if not os.path.isdir(buffer_zones_dir):
    run_create_buffer_zones(
        times=time_steps,
        rotation_model=rotation_model,
        topology_features=topology_features,
        static_polygons=static_polygons,
        anchor_plate_id=anchor_plate_id,
        output_dir=buffer_zones_dir,
        buffer_distance=buffer_distance, # Width of the buffer zones
        clip_to_overriding_plate=False,
        n_jobs=n_jobs,
        verbose=True,
        return_output=False,
    )

The cell below is interactive, allowing users to select a geological time and visualize buffer zones, seafloor age, and plate boundaries — including mid-ocean ridges and transform faults.

In [ ]:
@interact
def show_map(time=time_steps):
    gplot.time = time

    # Load seafloor age grid for the specified geological time
    agegrid_filename =  f"seafloor_age_{time}Ma.nc"
    agegrid_file = os.path.join(agegrid_dir, agegrid_filename)    
    agegrid = gplately.grids.read_netcdf_grid(agegrid_file)
    
    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(
        projection=projection,
        facecolor=(plt.cm.colors.to_rgba("darkgray", alpha=0.5)),
    )
    ax.set_global()

    # Plot seafloor age as background
    im = gplot.plot_grid(ax, agegrid.data, cmap=ccm.lapaz_r, vmin=0, vmax=230, alpha=0.7, zorder=1)
    # Plot continental areas in dark grey
    gplot.plot_coastlines(ax, facecolor="darkgray", edgecolor="none", zorder=2)
    # Plot plate motion vectors (arrows showing plate movement direction)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    # Load and plot buffer zones for the specified geological time
    buffer_zones_t_filename = f"buffer_zones_{time}Ma.geojson"
    buffer_zones_t_filename = os.path.join(buffer_zones_dir, buffer_zones_t_filename)
    buffer_zones_t = gpd.read_file(buffer_zones_t_filename)

    buffer_zones_t.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        facecolor="palegreen",
        edgecolor="none",
        alpha=0.7,
        zorder=4,
    )

    gplot.plot_topological_plate_boundaries(ax, color="dimgray", linewidth=1.2, zorder=5)
    gplot.plot_trenches(ax, color="black", zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color="black", zorder=7)
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=8)

    ax.text(0.505,-0.03, "60°E", transform=ax.transAxes, fontsize=16)
    ax.text(0.48,-0.03, "0°", transform=ax.transAxes, fontsize=16)
    ax.text(0.425,-0.027, "60°W", transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}
    
    cb = fig.colorbar(im, orientation="horizontal", shrink=0.4, pad=0.06, extend="max")
    cb.set_label("Seafloor age (Myr)", fontsize=16, labelpad=10)
    cb.set_ticks([0, 50, 100, 150, 200])
    cb.ax.tick_params(labelsize=16)
    
    # Dummy handle to trigger custom handler
    trench_handle = Line2D([], [], color="black")
    
    # Add custom handles
    custom_handles = [
        Patch(facecolor="darkgray", edgecolor="none"),
        Patch(facecolor="palegreen", edgecolor="none", alpha=0.7),
        trench_handle,
        Line2D([0], [0], color="dimgray", lw=1.2),
    ]
    custom_labels = [
        "Continental crust",
        "Target arc environment",
        "Subduction zone",
        "Plate boundary",
    ]

    # Draw legend
    ax.legend(custom_handles, custom_labels, fontsize=16, loc="lower left", bbox_to_anchor=(0, -0.25),
              handler_map={trench_handle: HandlerTrenchLine()})

    ax.set_title(f"{time} Ma", fontsize=25, y=1.04)
    
    plt.show()

### Reconstruct Mineral Occurrences

The following two cells reconstruct the locations of known mineral occurrences (porphyry) based on their ages stored in the input CSV file. This file must include the present-day longitude and latitude of each occurrence, its age, and an optional weight that reflects its significance. If the weight column is missing, equal weights are automatically assigned to all occurrences.

In [ ]:
# Reconstruct deposits if the file does not exist
if os.path.isfile(deposit_coords_recon_filename):
    deposit_coords_recon = pd.read_csv(deposit_coords_recon_filename)
else:
    deposit_coords_recon = prepare_deposit_data(
        deposit_data=deposit_coords_filename,
        buffer_zones_dir=buffer_zones_dir,
        rotation_model=rotation_model,
        topology_features=topology_features,
        static_polygons=static_polygons,
        anchor_plate_id=anchor_plate_id,
        output_filename=deposit_coords_recon_filename,
        time_steps=time_steps,
        min_time=time_min,
        max_time=time_max,
        n_jobs=n_jobs,
        verbose=True,
    )

In [ ]:
# Reconstruct all deposits for all time steps (for visualisation)
if os.path.isfile(deposit_coords_recon_all_filtered_filename):
    deposit_coords_recon_all = pd.read_csv(deposit_coords_recon_all_filename)
    deposit_coords_recon_all_filtered = pd.read_csv(deposit_coords_recon_all_filtered_filename)
else:
    deposit_coords_recon_all = partition_and_reconstruct(
        deposit_data=deposit_coords_filename,
        plate_reconstruction=plate_reconstruction,
        time_steps=time_steps,
        output_filename=deposit_coords_recon_all_filename,
        anchor_plate_id=anchor_plate_id,
        verbose=True,
    )

    deposit_coords_recon_all_filtered = filter_deposits_by_continents(
        input_csv=deposit_coords_recon_all_filename,
        output_csv=deposit_coords_recon_all_filtered_filename,
        continents_recon=continents_recon_filename,
        plate_reconstruction=plate_reconstruction,
        continents_recon_dir=continents_recon_dir,
        anchor_plate_id=anchor_plate_id,
        verbose=True,
    )

In [ ]:
# Create list of features available for plotting
subduction_data_columns = subduction_data.columns.tolist()
features_plot = subduction_data_columns.copy()
features_plot.remove("lon")
features_plot.remove("lat")
features_plot.remove("age (Ma)")
features_plot.remove("subducting_plate_ID")
features_plot.remove("trench_plate_ID")

The cell below is interactive, allowing users to select a geological time and plot reconstructed mineral occurrences. Seafloor age and plate boundaries — including mid-ocean ridges and transform faults — are also plotted.

In [ ]:
@interact
def show_map(time=time_steps, feature=features_plot):
    gplot.time = time

    # Filter mineral occurrences
    deposit_coords_recon_t = deposit_coords_recon_all_filtered[[f"lon_{time}", f"lat_{time}", "weight"]]
    deposit_coords_recon_t = deposit_coords_recon_t.dropna()

    # Load seafloor age grid for the specified geological time
    agegrid_filename =  f"seafloor_age_{time}Ma.nc"
    agegrid_file = os.path.join(agegrid_dir, agegrid_filename)    
    agegrid = gplately.grids.read_netcdf_grid(agegrid_file)
    
    if feature == "convergence_obliquity (degrees)":
        feat_min = -90
        feat_max = 90
    elif feature in [
        "convergence_rate (cm/yr)",
        "convergence_rate_orthogonal (cm/yr)",
        "slab_flux (m^2/yr)",
        "subduction_water_flux_lithosphere (t/m/yr)"
    ]:
        feat_min = 0
        feat_max = subduction_data[feature].quantile(0.99)
    else:
        feat_min = subduction_data[feature].quantile(0.01)
        feat_max = subduction_data[feature].quantile(0.99)
    
    features_t = subduction_data[subduction_data["age (Ma)"] == time]

    fig = plt.figure(figsize=(16, 12))
    ax = fig.add_axes(
        [0.1, 0.1, 0.8, 0.8],
        projection=projection,
        facecolor=(plt.cm.colors.to_rgba("darkgray", alpha=0.5)),
    )
    ax.set_global()

    # Create colour bar axes for the selected feature and seafloor age
    cax_feat = fig.add_axes([0.33, 0.17, 0.25, 0.02])
    cax_bg = fig.add_axes([0.62, 0.17, 0.25, 0.02])

    # Plot seafloor age as background
    bg = gplot.plot_grid(ax, agegrid.data, cmap=ccm.lapaz_r, vmin=0, vmax=230, alpha=0.7, zorder=1)
    # Plot continental areas in dark grey
    gplot.plot_coastlines(ax, facecolor="darkgray", edgecolor="none", zorder=2)
    # Plot plate motion vectors (arrows showing plate movement direction)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    # Plot feature values along trench lines
    sc0 = ax.scatter(features_t["lon"], features_t["lat"], 100, marker=".",
                     c=features_t[feature], cmap=ccm.hawaii_r, vmin=feat_min, vmax=feat_max,
                     transform=ccrs.PlateCarree(), zorder=4)
    
    gplot.plot_topological_plate_boundaries(ax, color="dimgray", linewidth=1.2, zorder=5)
    gplot.plot_trenches(ax, color="black", alpha=0.5, zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color="black", alpha=0.5, zorder=7)

    # Plot reconstructed mineral occurrences
    sc1 = ax.scatter(
        deposit_coords_recon_t[f"lon_{time}"],
        deposit_coords_recon_t[f"lat_{time}"],
        transform=ccrs.PlateCarree(),
        marker="o",
        facecolor="yellow",
        edgecolor="black",
        s = [w * 10 for w in deposit_coords_recon_t["weight"]],
        alpha=0.7,
        zorder=8
    )
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=9)

    ax.text(0.505,-0.03, "60°E", transform=ax.transAxes, fontsize=16)
    ax.text(0.48,-0.03, "0°", transform=ax.transAxes, fontsize=16)
    ax.text(0.425,-0.027, "60°W", transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}

    # Create colour bars with labels
    cbar_feat = fig.colorbar(sc0, cax=cax_feat, orientation="horizontal", extend="max")
    cbar_feat.set_label(format_feature_name(feature), fontsize=16, labelpad=10)
    cbar_feat.ax.tick_params(labelsize=16)

    cbar_bg = fig.colorbar(bg, cax=cax_bg, orientation="horizontal", extend="max")
    cbar_bg.set_label("Seafloor age (Myr)", fontsize=16, labelpad=10)
    cbar_bg.set_ticks([0, 50, 100, 150, 200])
    cbar_bg.ax.tick_params(labelsize=16)
    
    # Dummy handle to trigger custom handler
    trench_handle = Line2D([], [], color="black")
    
    # Add custom handles
    custom_handles = [
        Patch(facecolor="darkgray", edgecolor="none"),
        trench_handle,
        Line2D([0], [0], color="dimgray", lw=1.2),
        Line2D([0], [0], marker="o", markerfacecolor="yellow", markeredgecolor="black", markersize=15, linestyle="None"),
    ]
    custom_labels = [
        "Continental crust",
        "Subduction zone",
        "Plate boundary",
        "Mineral occurrence",
    ]

    # Draw legend
    ax.legend(custom_handles, custom_labels, fontsize=16, loc="lower left", bbox_to_anchor=(0, -0.22),
              handler_map={trench_handle: HandlerTrenchLine()})
    
    ax.set_title(f"{time} Ma", fontsize=25, y=1.04)
    
    plt.show()

### Unlabelled Samples

To train a supervised machine learning model for predicting mineralisation probability in our target areas, we need both positive samples (known mineral occurrences) and negative samples (representing non-mineralised areas). However, creating a reliable set of negative samples is challenging.

To address this, we generate a set of unlabelled samples that are later classified as either positive or negative using positive and unlabelled bagging method. We remove any unlabelled samples identified as positive and retain only the negatives for model training.

For each time step, we generate uniformly distributed unlabelled samples within the target areas (previously defined buffer zones). The number of generated samples can be controlled with the `num` argument in the `generate_unlabelled_points` function or by setting the corresponding parameter in the `parameters.py` file.

In [ ]:
# Generate unlabelled samples if the relevant file does not exist
if os.path.isfile(unlabelled_coords_filename):
    unlabelled_coords = pd.read_csv(unlabelled_coords_filename)
else:
    unlabelled_coords = generate_unlabelled_points(
        times=time_steps,
        input_dir=buffer_zones_dir,
        num=num_random, # Number of random points to generate
        seed=42, # For reproducible random sampling
        rotation_model=rotation_model,
        topology_features=topology_features,
        static_polygons=static_polygons,
        anchor_plate_id=anchor_plate_id,
        output_filename=unlabelled_coords_filename,
        n_jobs=n_jobs,
        verbose=True,
    )

The cell below is interactive, allowing users to select a geological time and plot unlabelled samples. Seafloor age and plate boundaries — including mid-ocean ridges and transform faults — are also plotted.

In [ ]:
@interact
def show_map(time=time_steps):
    gplot.time = time

    # Load seafloor age grid for the specified geological time
    agegrid_filename =  f"seafloor_age_{time}Ma.nc"
    agegrid_file = os.path.join(agegrid_dir, agegrid_filename)    
    agegrid = gplately.grids.read_netcdf_grid(agegrid_file)
    
    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(
        projection=projection,
        facecolor=(plt.cm.colors.to_rgba("silver", alpha=0.5))
    )
    ax.set_global()

    # Plot seafloor age as background
    im = gplot.plot_grid(ax, agegrid.data, cmap=ccm.lapaz_r, vmin=0, vmax=230, alpha=0.7, zorder=1)
    # Plot continental areas in dark grey
    gplot.plot_coastlines(ax, facecolor="darkgray", edgecolor="none", zorder=2)
    # Plot plate motion vectors (arrows showing plate movement direction)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    # Load and plot buffer zones for the specified geological time
    buffer_zones_t_filename = f"buffer_zones_{time}Ma.geojson"
    buffer_zones_t_filename = os.path.join(buffer_zones_dir, buffer_zones_t_filename)
    buffer_zones_t = gpd.read_file(buffer_zones_t_filename)

    buffer_zones_t.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        facecolor="palegreen",
        edgecolor="none",
        alpha=0.7,
        zorder=4,
    )
    
    unlabelled_coords_t = unlabelled_coords[unlabelled_coords["age (Ma)"] == time]

    gplot.plot_topological_plate_boundaries(ax, color="dimgray", linewidth=1.2, zorder=5)
    gplot.plot_trenches(ax, color="black", zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color="black", zorder=7)

    # Plot unlabelled samples
    ax.scatter(
        unlabelled_coords_t["lon"],
        unlabelled_coords_t["lat"],
        transform=ccrs.PlateCarree(),
        marker="X",
        facecolor="cyan",
        edgecolor="black",
        s=50,
        zorder=8
    )

    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=9)

    ax.text(0.505,-0.03, "60°E", transform=ax.transAxes, fontsize=16)
    ax.text(0.48,-0.03, "0°", transform=ax.transAxes, fontsize=16)
    ax.text(0.425,-0.027, "60°W", transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}

    cb = fig.colorbar(im, orientation="horizontal", shrink=0.4, pad=0.06, extend="max")
    cb.set_label("Seafloor age (Myr)", fontsize=16, labelpad=10)
    cb.set_ticks([0, 50, 100, 150, 200])
    cb.ax.tick_params(labelsize=16)
    
    # Dummy handle to trigger custom handler
    trench_handle = Line2D([], [], color="black")
    
    # Add custom handles
    custom_handles = [
        Patch(facecolor="darkgray", edgecolor="none"),
        Patch(facecolor="palegreen", edgecolor="none", alpha=0.7),
        trench_handle,
        Line2D([0], [0], color="dimgray", lw=1.2),
        Line2D([0], [0], marker="X", markerfacecolor="cyan", markeredgecolor="black", markersize=10, linestyle="None"),
    ]
    custom_labels = [
        "Continental crust",
        "Target arc environment",
        "Subduction zone",
        "Plate boundary",
        "Random sample",
    ]

    # Draw legend
    ax.legend(custom_handles, custom_labels, fontsize=16, loc="lower left", bbox_to_anchor=(0, -0.25),
              handler_map={trench_handle: HandlerTrenchLine()})

    ax.set_title(f"{time} Ma", fontsize=25, y=1.04)
        
    plt.show()

### Back-Arc Points

The cell below generates a grid of points at a user-defined resolution within previously generated buffer zones. You can adjust the resolution either by modifying the `resolution` argument in the function or by updating the value in the `parameters.py` file. The machine learning model will then be used to predict mineralisation at these locations.

In [ ]:
# Generate grid points if the relevant file does not exist
if os.path.isfile(backarc_coords_filename):
    backarc_coords = pd.read_csv(backarc_coords_filename)
    backarc_coords = backarc_coords.dropna(subset=["present_lon", "present_lat"])
else:
    backarc_coords = generate_grid_points(
        times=time_steps,
        resolution=grid_resolution,
        polygons_dir=buffer_zones_dir,
        rotation_model=rotation_model,
        topology_features=topology_features,
        static_polygons=static_polygons,
        anchor_plate_id=anchor_plate_id,
        output_filename=backarc_coords_filename,
        n_jobs=n_jobs,
        verbose=True,
    )
    
    backarc_coords = backarc_coords.dropna(subset=["present_lon", "present_lat"])

The cell below is interactive, allowing users to select a geological time and plot grid points generated within the previously generated buffer zones. Seafloor age and plate boundaries — including mid-ocean ridges and transform faults — are also plotted.

In [ ]:
@interact
def show_map(time=time_steps):
    gplot.time = time    

    # Load seafloor age grid for the specified geological time
    agegrid_filename =  f"seafloor_age_{time}Ma.nc"
    agegrid_file = os.path.join(agegrid_dir, agegrid_filename)    
    agegrid = gplately.grids.read_netcdf_grid(agegrid_file)
    
    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(
        projection=projection,
        facecolor=(plt.cm.colors.to_rgba("darkgray", alpha=0.5))
    )
    ax.set_global()

    # Plot seafloor age as background
    im = gplot.plot_grid(ax, agegrid.data, cmap=ccm.lapaz_r, vmin=0, vmax=230, alpha=0.7, zorder=1)
    # Plot continental areas in dark grey
    gplot.plot_coastlines(ax, facecolor="darkgray", edgecolor="none", zorder=2)
    # Plot plate motion vectors (arrows showing plate movement direction)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    # Load and plot buffer zones for the specified geological time
    # buffer_zones_t_filename = f"buffer_zones_{time}Ma.geojson"
    # buffer_zones_t_filename = os.path.join(buffer_zones_dir, buffer_zones_t_filename)
    # buffer_zones_t = gpd.read_file(buffer_zones_t_filename)

    # buffer_zones_t.plot(
    #     ax=ax,
    #     transform=ccrs.PlateCarree(),
    #     facecolor="palegreen",
    #     edgecolor="none",
    #     alpha=0.7,
    #     zorder=4,
    # )

    # Plot grid points
    backarc_coords_t = backarc_coords[backarc_coords["age (Ma)"] == time]

    gplot.plot_topological_plate_boundaries(ax, color="dimgray", linewidth=1.2, zorder=5)
    
    ax.scatter(
        backarc_coords_t["lon"],
        backarc_coords_t["lat"],
        transform=ccrs.PlateCarree(),
        marker=".",
        c="red",
        s=1,
        alpha=0.5,
        zorder=6
    )
    
    gplot.plot_trenches(ax, color="black", zorder=7)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color="black", zorder=8)

    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=9)

    ax.text(0.505,-0.03, "60°E", transform=ax.transAxes, fontsize=16)
    ax.text(0.48,-0.03, "0°", transform=ax.transAxes, fontsize=16)
    ax.text(0.425,-0.027, "60°W", transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}
    
    cb = fig.colorbar(im, orientation="horizontal", shrink=0.4, pad=0.06, extend="max")
    cb.set_label("Seafloor age (Myr)", fontsize=16, labelpad=10)
    cb.set_ticks([0, 50, 100, 150, 200])
    cb.ax.tick_params(labelsize=16)
    
    # Dummy handle to trigger custom handler
    trench_handle = Line2D([], [], color="black")
    
    # Add custom handles
    custom_handles = [
        Patch(facecolor="darkgray", edgecolor="none"),
        # Patch(facecolor="palegreen", edgecolor="none", alpha=0.7),
        trench_handle,
        Line2D([0], [0], color="dimgray", lw=1.2),
        Line2D([0], [0], marker=".", markerfacecolor="red", markeredgecolor="none", markersize=10, linestyle="None")
    ]
    custom_labels = [
        "Continental crust",
        # "Target arc environment",
        "Subduction zone",
        "Plate boundary",
        "Target point within\narc environment",
    ]

    # Draw legend
    ax.legend(custom_handles, custom_labels, fontsize=16, loc="lower left", bbox_to_anchor=(0, -0.25),
              handler_map={trench_handle: HandlerTrenchLine()})

    ax.set_title(f"{time} Ma", fontsize=25, y=1.04)
        
    plt.show()

### Coregistration

This code block performs co-registration by finding the closest trench point to each point in the previously created point sets — including reconstructed mineral occurrences, unlabelled samples, and grid points within the buffer. It then assigns the feature values of the nearest trench point to each of these points, along with the distance between them. The `run_coregister_crustal_thickness` function is specifically designed to coregister points with crustal thickness grids. In the final step, missing values within the point sets are handled using the `fill_missing_values` function, which supports two imputation strategies: iterative imputation and median-value imputation. Because iterative imputation is computationally expensive, it is applied only to mineral occurrence data and unlabeled samples. Missing values in grid points are instead filled using median values. For additional implementation details, refer to the function documentation.

In [ ]:
if not os.path.isfile(deposit_data_filename):
    deposit_data = run_coregister_point_data(
        point_data=deposit_coords_recon,
        subduction_data=subduction_data,
        n_jobs=n_jobs,
        verbose=True,
    )
    
    deposit_data = run_coregister_crustal_thickness(
        point_data=deposit_data,
        input_dir=crusthick_dir,
        n_jobs=n_jobs,
        verbose=True,
    )

    # Fill in missing values
    deposit_data = fill_missing_values(
        deposit_data,
        method="iterative",
        exclude_columns=columns_to_drop_deposit,
        # distance_threshold=distance_threshold,
        report_missing=True,
        random_state=42,
        export_to_csv=True,
        output_path=deposit_data_filename,
        n_jobs=n_jobs,
        verbose=True
    )

if not os.path.isfile(unlabelled_data_filename):
    unlabelled_data = run_coregister_point_data(
        point_data=unlabelled_coords,
        subduction_data=subduction_data,
        n_jobs=n_jobs,
        verbose=True,
    )
    
    unlabelled_data = run_coregister_crustal_thickness(
        point_data=unlabelled_data,
        input_dir=crusthick_dir,
        n_jobs=n_jobs,
        verbose=True,
    )

    # Fill in missing values
    unlabelled_data = fill_missing_values(
        unlabelled_data,
        method="iterative",
        exclude_columns=columns_to_drop_unlabelled,
        # distance_threshold=distance_threshold,
        report_missing=True,
        random_state=42,
        export_to_csv=True,
        output_path=unlabelled_data_filename,
        n_jobs=n_jobs,
        verbose=True
    )

if not os.path.isfile(backarc_data_filename):
    backarc_data = run_coregister_point_data(
        point_data=backarc_coords,
        subduction_data=subduction_data,
        n_jobs=n_jobs,
        verbose=True,
    )
    
    backarc_data = run_coregister_crustal_thickness(
        point_data=backarc_data,
        input_dir=crusthick_dir,
        n_jobs=n_jobs,
        verbose=True,
    )

    # Fill in missing values
    backarc_data = fill_missing_values(
        backarc_data,
        method="median",
        exclude_columns=columns_to_drop_backarc,
        # distance_threshold=distance_threshold,
        report_missing=True,
        random_state=42,
        export_to_csv=True,
        output_path=backarc_data_filename,
        n_jobs=n_jobs,
        verbose=True
    )